[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# A Frost Forecast &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' labels and chosen models, rebuilt here from the same made-up stations so that each
answer runs on its own.

In [2]:
frost_train, frost_valid, frost_test = [(rows["tmin"] < 0).astype(int) for rows in (train, valid, test)]


def lightgbm_with(leaves):
    model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=leaves, random_state=0, verbose=-1)
    return model.fit(train[FEATURES], frost_train, eval_set=[(valid[FEATURES], frost_valid)],
                     callbacks=[lgb.early_stopping(100, verbose=False)])


lightgbm_frost = lightgbm_with(31)
xgboost_frost = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=6, early_stopping_rounds=100,
                                  eval_metric="logloss", random_state=0).fit(
    train[FEATURES], frost_train, eval_set=[(valid[FEATURES], frost_valid)], verbose=False)

**1.** Score the chosen LightGBM model on the validation stations at thresholds of 0.3, 0.5 and 0.7, and print the accuracy and the number of frost days it misses at each.

In [3]:
probability = lightgbm_frost.predict_proba(valid[FEATURES])[:, 1]
for threshold in (0.3, 0.5, 0.7):
    decided = probability > threshold
    missed = int(((frost_valid == 1) & ~decided).sum())
    print(threshold, "accuracy", round(accuracy_score(frost_valid, decided), 3), "| frost days missed", missed)

0.3 accuracy 0.949 | frost days missed 48
0.5 accuracy 0.947 | frost days missed 80
0.7 accuracy 0.94 | frost days missed 116


On these made-up stations, a threshold of 0.3 misses 48 frost days, against 80 at 0.5 and 116 at 0.7, while the accuracy moves only from 0.949 to 0.947 and 0.94. A road authority that fears ice more than wasted salt would take a lower threshold, and would choose it on the validation stations, as here.

**2.** Add a candidate with 15 leaves to LightGBM's comparison, trained the same way, and print its number of trees and its validation log loss beside the three in the worked examples.

In [4]:
for leaves in (7, 15, 31, 127):
    model = lightgbm_with(leaves)
    print(leaves, "leaves | trees", model.best_iteration_,
          "| log loss", round(log_loss(frost_valid, model.predict_proba(valid[FEATURES])[:, 1]), 3))

7 leaves | trees 507 | log loss 0.135
15 leaves | trees 267 | log loss 0.13
31 leaves | trees 178 | log loss 0.131
127 leaves | trees 126 | log loss 0.138


On these made-up stations the 15-leaf candidate keeps 267 trees and reaches a validation log loss of 0.13, which is 0.130 to three places, just below 0.131 for 31 leaves, so with four candidates the choice would move to 15 leaves. The gap is 0.001, smaller than the gaps between the original three, and one split of the stations cannot say whether it would hold on another.

**3.** Group the validation days by the chosen XGBoost model's probabilities, in the same bands as the worked examples, and print each band's average probability and share of frost days.

In [5]:
probability = xgboost_frost.predict_proba(valid[FEATURES])[:, 1]
band = pd.cut(probability, [0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
checked = pd.DataFrame({"probability": probability, "frost": frost_valid.to_numpy()})
print(checked.groupby(band, observed=True).agg(days=("frost", "size"), forecast=("probability", "mean"),
                                                happened=("frost", "mean")).round(3))

               days  forecast  happened
(-0.001, 0.2]  1114     0.026     0.032
(0.2, 0.4]       67     0.303     0.418
(0.4, 0.6]       50     0.492     0.540
(0.6, 0.8]       78     0.709     0.718
(0.8, 1.0]     1191     0.984     0.984


On these made-up stations XGBoost's groups follow the same pattern as LightGBM's. The two outer groups match within 0.006, and in the groups from 0.2 to 0.4 and from 0.4 to 0.6 frost came more often than forecast, 0.418 against 0.303 and 0.540 against 0.492.

**4.** Ask both loaded models about the same planned night with a cloud cover of 1.0 and a wind of 8 meters a second, and print the two chances of frost beside the clear, calm night's.

In [6]:
lightgbm_frost.booster_.save_model(scratch / "frost_lightgbm.txt")
xgboost_frost.save_model(scratch / "frost_xgboost.json")
lightgbm_loaded = lgb.Booster(model_file=scratch / "frost_lightgbm.txt")
xgboost_loaded = xgb.XGBClassifier()
xgboost_loaded.load_model(scratch / "frost_xgboost.json")

for cloud, wind in ((0.0, 0.5), (1.0, 8.0)):
    planned = pd.DataFrame({
        "latitude": [62.0], "elevation_m": [250], "coast_km": [60.0],
        "terrain": pd.Categorical(["valley"], categories=TERRAINS),
        "day_of_year": [120], "cloud_cover": [cloud], "wind_ms": [wind], "snow_cm": [0.0],
    })
    print("cloud", cloud, "wind", wind, "| LightGBM", round(float(lightgbm_loaded.predict(planned[FEATURES])[0]), 3),
          "| XGBoost", round(float(xgboost_loaded.predict_proba(planned[FEATURES])[0, 1]), 3))

cloud 0.0 wind 0.5 | LightGBM 0.856 | XGBoost 0.818
cloud 1.0 wind 8.0 | LightGBM 0.025 | XGBoost 0.03


On these made-up stations, a cloudy, windy version of the same night drops the chance of frost from 0.856 to 0.025 in LightGBM and from 0.818 to 0.03 in XGBoost. Both models have learned from the made-up data that a cloudy, windy night is far less likely to freeze.

In [7]:
shutil.rmtree(scratch)
print("scratch removed:", not scratch.exists())

scratch removed: True


---

&#8592; **Back to:** [A Frost Forecast](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/15-a-frost-forecast.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
